# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fasiuu/FLYRANK_TASK1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
# Setup — shared feature builder from this repo + DuckDB connection to the hosted release.
# Token: Colab Secret HF_TOKEN (preferred) or a getpass prompt. Never paste a token into a cell.
%pip -q install duckdb
import os, sys, subprocess, json, warnings
warnings.filterwarnings('ignore')
if not os.path.exists('FLYRANK_TASK1'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/Fasiuu/FLYRANK_TASK1.git'], check=True)
sys.path.insert(0, 'FLYRANK_TASK1/work/scripts')
import numpy as np, pandas as pd
import flyrank_features as ff
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
np.random.seed(ff.SEED)
con = ff.connect()
os.makedirs('work/outputs', exist_ok=True)
print('connected; seed =', ff.SEED)

Hugging Face READ token (hf_...): ··········
connected; seed = 42


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule in plain words:** a page is worth reviewing when it's already losing impressions, its rank is slipping, last month was propped up by one day, it's past the "decay cliff" age, or it's only visible on some days. Each sign is one point. Ties go to the more visible page.

| reason code | condition |
|---|---|
| `sliding_impressions` | impressions in the last 30 days < 85% of the 30 days before |
| `position_slipping` | average position got more than 1 place worse |
| `one_day_spike` | the biggest single day is more than 25% of the 30-day total |
| `aging_page` | older than 270 days (FlyRank paper's decay point) |
| `patchy_visibility` | impressions on fewer than 15 of the last 30 days |

I wrote this from the starter's reason codes and the paper's advice, *before* the model work. It stays frozen from here on.

In [5]:
# Build the per-item panel for the two development decision dates (one DuckDB scan, ~5-8 min).
import time
t0 = time.time()
panel = ff.build_panel(con, ff.DEV_DATES)
panel['decision_date'] = pd.to_datetime(panel['decision_date']).dt.strftime('%Y-%m-%d')
m = ff.model_frame(panel)
print(f'{len(panel):,} item-date rows built in {time.time()-t0:.0f}s; {len(m):,} pass the population checks')
# Validation design used everywhere: client-grouped AND time-forward.
# Each client gets a fixed fold id (a hash of its pseudonym). To score fold f at decision date T2,
# a model is trained on the OTHER folds' clients at T1 only - so it has never seen the client,
# and never seen the outcome month it is judged on.
T1, T2 = ff.DEV_DATES
va = m[m.decision_date == T2].copy()
va['fold'] = va.client_hash_id.map(ff.client_fold)
print(f'judge snapshot {T2}: {len(va):,} rows, {va.client_hash_id.nunique()} clients, base rate {va.declined.mean():.3f}')
print('rows per client fold:', va.fold.value_counts().sort_index().to_dict())

def baseline(df):
    """Frozen transparent rule: one point per warning sign, ties broken by current visibility."""
    r = pd.DataFrame(index=df.index)
    r['sliding_impressions'] = df.imp_last30 < 0.85 * df.imp_prev30      # already losing impressions
    r['position_slipping']   = df.pos_change > 1.0                        # average rank got >1 place worse
    r['one_day_spike']       = df.spike_share > 0.25                      # month propped up by a single day
    r['aging_page']          = df.content_age_days > 270                  # past the decay cliff in FlyRank's paper
    r['patchy_visibility']   = df.days_with_imp_last30 < 15               # visible on fewer than half the days
    points = r.sum(axis=1)
    score = points + df.log_imp_last30 / 100.0
    reasons = r.apply(lambda row: '|'.join(c for c in r.columns if row[c]) or 'none', axis=1)
    return score, reasons, r
for c in ['sliding_impressions', 'position_slipping', 'one_day_spike', 'aging_page', 'patchy_visibility']:
    print('-', c)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

207,726 item-date rows built in 137s; 134,757 pass the population checks
judge snapshot 2026-04-30: 71,297 rows, 27 clients, base rate 0.485
rows per client fold: {0: 30108, 1: 9166, 2: 8019, 3: 24004}
- sliding_impressions
- position_slipping
- one_day_spike
- aging_page
- patchy_visibility


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The rule scores every page at the 2026-04-30 snapshot (outcome month: May), ranks them, and writes `work/outputs/baseline_action_score.csv`. The CSV is gitignored; the metrics go to `baseline_metrics.json`.

The rule clearly beats random picking at the top of the list (precision@50 ≈ 0.88 against a base rate ≈ 0.49). But its ROC-AUC is only about 0.61, so below the very top its ordering is weak.

The per-sign table explains why. `sliding_impressions` does real work (decline rate 64% when it fires vs 30% when not), and `one_day_spike` and `patchy_visibility` are strong but rare. **`aging_page` points the wrong way**: pages it flags fall behind *less* often (44% vs 53%). That matches the signal audit. I keep the rule frozen anyway, because the point of a baseline is to be honest, not tuned.

In [6]:
from sklearn.metrics import roc_auc_score, average_precision_score
def evaluate(scores, y, ks=(20, 50, 100, 500)):
    y = np.asarray(y).astype(int)
    d = {'base_rate': round(float(y.mean()), 4), 'auc': round(float(roc_auc_score(y, scores)), 4),
         'ap': round(float(average_precision_score(y, scores)), 4)}
    for k in ks:
        d[f'p@{k}'] = round(ff.precision_at_k(scores, y, k), 4)
    return d
def client_queue_at_10(df, scores, min_items=30):
    """How a single site would use it: precision of its OWN top-10, averaged over clients
    (with those clients' average base rate next to it)."""
    tmp = df[['client_hash_id', 'declined']].copy(); tmp['s'] = np.asarray(scores)
    rows = [(g.sort_values('s', ascending=False).declined.head(10).mean(), g.declined.mean())
            for _, g in tmp.groupby('client_hash_id') if len(g) >= min_items]
    return {'client_p@10': round(float(np.mean([a for a, b in rows])), 3),
            'client_base': round(float(np.mean([b for a, b in rows])), 3), 'clients': len(rows)}

score, reasons, flags = baseline(va)
queue = va.assign(baseline_score=score.round(3), reason_codes=reasons).sort_values('baseline_score', ascending=False)
queue['rank'] = np.arange(1, len(queue) + 1)
cols_out = ['rank', 'decision_date', 'client_hash_id', 'content_hash_id', 'baseline_score', 'reason_codes',
            'imp_last30', 'imp_prev30', 'pos_last30', 'pos_change', 'spike_share', 'content_age_days']
queue[cols_out].to_csv('work/outputs/baseline_action_score.csv', index=False)       # gitignored: stays local
res = evaluate(score, va.declined); res.update(client_queue_at_10(va, score))
json.dump(res, open('work/outputs/baseline_metrics.json', 'w'), indent=2)
print('wrote work/outputs/baseline_action_score.csv', queue.shape)
print(json.dumps(res, indent=2))
print('\nhow often each warning sign fires, and the decline rate with / without it:')
for c in flags.columns:
    print(f'{c:22} fires on {flags[c].mean():.3f} | decline rate when fired {va.declined[flags[c]].mean():.3f} | when not {va.declined[~flags[c]].mean():.3f}')

wrote work/outputs/baseline_action_score.csv (71297, 48)
{
  "base_rate": 0.4847,
  "auc": 0.6139,
  "ap": 0.5848,
  "p@20": 0.9,
  "p@50": 0.88,
  "p@100": 0.89,
  "p@500": 0.884,
  "client_p@10": 0.811,
  "client_base": 0.498,
  "clients": 19
}

how often each warning sign fires, and the decline rate with / without it:
sliding_impressions    fires on 0.540 | decline rate when fired 0.642 | when not 0.300
position_slipping      fires on 0.493 | decline rate when fired 0.539 | when not 0.432
one_day_spike          fires on 0.024 | decline rate when fired 0.770 | when not 0.478
aging_page             fires on 0.495 | decline rate when fired 0.443 | when not 0.526
patchy_visibility      fires on 0.015 | decline rate when fired 0.877 | when not 0.479


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top 20, each with the action I'd suggest, a confidence note, and the actual outcome in May. "What would make it wrong" for these pages: the drop was a spike reverting (nothing to fix), the whole site moved (already handled by the label), the traffic moved to a sibling page (consolidation), or seasonality.

In [7]:
def action_for(codes):
    if 'one_day_spike' in codes: return 'monitor: check what caused the spike before editing'
    if 'position_slipping' in codes and 'sliding_impressions' in codes: return 'review: compare with the competing results, refresh'
    if 'aging_page' in codes: return 'refresh: update facts, examples, internal links'
    if 'patchy_visibility' in codes: return 'monitor: visibility too thin to act on'
    return 'review'
top = queue.head(20).copy()
top['action'] = top.reason_codes.map(action_for)
top['confidence'] = np.where(top.reason_codes.str.count(r'\|') >= 2, 'medium (3+ signs)', 'low (1-2 signs)')
top['outcome'] = np.where(top.declined == 1, 'DECLINED vs site', 'held')
print(top[['rank', 'reason_codes', 'action', 'confidence', 'imp_prev30', 'imp_last30', 'imp_next30', 'outcome']].to_string(index=False))
print(f"\ntop-20 hit rate {top.declined.mean():.2f} vs base rate {va.declined.mean():.3f}")

 rank                                                                     reason_codes                                              action        confidence  imp_prev30  imp_last30  imp_next30          outcome
    1 sliding_impressions|position_slipping|one_day_spike|aging_page|patchy_visibility monitor: check what caused the spike before editing medium (3+ signs)       396.0       306.0         1.0 DECLINED vs site
    2 sliding_impressions|position_slipping|one_day_spike|aging_page|patchy_visibility monitor: check what caused the spike before editing medium (3+ signs)      1162.0       153.0         2.0 DECLINED vs site
    3 sliding_impressions|position_slipping|one_day_spike|aging_page|patchy_visibility monitor: check what caused the spike before editing medium (3+ signs)       556.0       111.0        32.0 DECLINED vs site
    4 sliding_impressions|position_slipping|one_day_spike|aging_page|patchy_visibility monitor: check what caused the spike before editing medium (3+ signs)    

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks.** The misses at the top tend to be pages that slid last month and then bounced back, so momentum alone over-reacts to a one-month dip. The top 50 also leans heavily on a few large clients, because the tie-break favours visible pages. Per-site queues fix that, so w05 and w07 rank within each site.

**Leakage check.** The rule reads seven columns, all from on or before T. Nothing comes from the outcome month, and no product flag is involved (the release has none).

In [8]:
weak = top[top.declined == 0]
print(f'{len(weak)} of the top 20 did NOT fall behind their site. What they look like:')
print(weak[['rank', 'reason_codes', 'imp_prev30', 'imp_last30', 'imp_next30', 'client_ratio', 'spike_share']].round(2).to_string(index=False))
print('\nclient concentration of the top 50:', queue.head(50).client_hash_id.value_counts().head(5).to_dict())
used = ['imp_last30', 'imp_prev30', 'pos_change', 'spike_share', 'content_age_days', 'days_with_imp_last30', 'log_imp_last30']
print('\nleakage check - inputs the rule reads:', used)
print('any computed from the outcome month?', any('next' in c or 'ratio' in c for c in used))
print('product flags / scores used?', 'none - the release ships none, and the rule is hand-written')

2 of the top 20 did NOT fall behind their site. What they look like:
 rank                                                                     reason_codes  imp_prev30  imp_last30  imp_next30  client_ratio  spike_share
   18 sliding_impressions|position_slipping|one_day_spike|aging_page|patchy_visibility       164.0        47.0        79.0          0.55         0.26
   19 sliding_impressions|position_slipping|one_day_spike|aging_page|patchy_visibility       208.0        48.0        91.0          0.55         0.29

client concentration of the top 50: {'client_ff644d8251367cbb': 19, 'client_9958f0a7ae1df715': 12, 'client_62f4a7e64f5e0096': 6, 'client_c182d11e4862a37d': 3, 'client_65de48885f4ef01b': 3}

leakage check - inputs the rule reads: ['imp_last30', 'imp_prev30', 'pos_change', 'spike_share', 'content_age_days', 'days_with_imp_last30', 'log_imp_last30']
any computed from the outcome month? False
product flags / scores used? none - the release ships none, and the rule is hand-written

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.